## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 11: Autoencoder
Niveau: Anfänger
Aufgabe 11.1: Einfacher Autoencoder – Encoder und Decoder

Lernziel: Autoencoder-Architektur und Prinzip verstehen
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, _), (X_test, _) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

print("=== Autoencoder Konzept ===")
print("Encoder: 784 → 128 → 64 → 32 (Bottleneck)")
print("Decoder: 32 → 64 → 128 → 784 (Rekonstruktion)")

# Autoencoder
LATENT_DIM = 32

eingabe  = keras.Input(shape=(784,))
# Encoder
x = keras.layers.Dense(128, activation='relu')(eingabe)
x = keras.layers.Dense(64, activation='relu')(x)
latent = keras.layers.Dense(LATENT_DIM, activation='relu', name='latent')(x)
# Decoder
x = keras.layers.Dense(64, activation='relu')(latent)
x = keras.layers.Dense(128, activation='relu')(x)
ausgabe = keras.layers.Dense(784, activation='sigmoid')(x)

autoencoder = keras.Model(eingabe, ausgabe, name='Autoencoder')
encoder     = keras.Model(eingabe, latent, name='Encoder')

autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.summary()

history = autoencoder.fit(X_train, X_train,
                           epochs=10,
                           batch_size=256,
                           validation_split=0.1,
                           verbose=1)

# Rekonstruktionen visualisieren
X_rekonstruiert = autoencoder.predict(X_test[:10], verbose=0)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    axes[0, i].imshow(X_test[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title('Original')
    
    axes[1, i].imshow(X_rekonstruiert[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('Rekonstruiert')

plt.suptitle(f'Autoencoder Rekonstruktionen (Bottleneck={LATENT_DIM})')
plt.tight_layout()
plt.savefig('autoencoder_rekonstruktion.png', dpi=150)
plt.show()

# Kompressionsverhältnis
print(f"\nOriginal Dimension: 784")
print(f"Latent Dimension:   {LATENT_DIM}")
print(f"Kompressionsrate:   {784/LATENT_DIM:.1f}:1")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 11: Autoencoder
Niveau: Anfänger
Aufgabe 11.2: Dimensionalitätsreduktion mit Autoencoder

Lernziel: Autoencoder als Dimensionsreduktion und Visualisierung
Datensatz: MNIST
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 784).astype('float32') / 255.0

# 2D Autoencoder für Visualisierung
eingabe = keras.Input(shape=(784,))
x = keras.layers.Dense(128, activation='relu')(eingabe)
x = keras.layers.Dense(64, activation='relu')(x)
latent_2d = keras.layers.Dense(2, name='latent_2d')(x)  # 2D!
x = keras.layers.Dense(64, activation='relu')(latent_2d)
x = keras.layers.Dense(128, activation='relu')(x)
ausgabe = keras.layers.Dense(784, activation='sigmoid')(x)

ae_2d = keras.Model(eingabe, ausgabe)
encoder_2d = keras.Model(eingabe, latent_2d)

ae_2d.compile(optimizer='adam', loss='mse')
ae_2d.fit(X_train, X_train, epochs=15, batch_size=256, verbose=0)

# Latent Space Visualisierung
X_latent = encoder_2d.predict(X_test[:3000], verbose=0)
y_latent = y_test[:3000]

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_latent[:, 0], X_latent[:, 1],
                       c=y_latent, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, label='Ziffer')
plt.title('2D Latent Space (Autoencoder) – MNIST')
plt.xlabel('Latente Dimension 1')
plt.ylabel('Latente Dimension 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('autoencoder_latent_space.png', dpi=150)
plt.show()

print("Ähnliche Ziffern sollten nahe beieinander liegen!")
print("Vergleich mit PCA:")

from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_test[:3000])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1],
                       c=y_latent, cmap='tab10', alpha=0.6, s=10)
plt.colorbar(scatter, label='Ziffer')
plt.title('2D PCA – MNIST')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pca_latent_space.png', dpi=150)
plt.show()


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
educx GmbH – Modul 3: Neuronale Netze
Tag 11: Autoencoder
Niveau: Anfänger
Aufgabe 11.3: Rauschunterdrückung mit Denoising Autoencoder

Lernziel: Autoencoder zur Bildentrauschung verwenden
Datensatz: MNIST (verrauscht)
"""

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

(X_train, _), (X_test, _) = keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test  = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# Rauschen hinzufügen
RAUSCH_FAKTOR = 0.4
X_train_rausch = np.clip(X_train + RAUSCH_FAKTOR * np.random.randn(*X_train.shape), 0, 1).astype('float32')
X_test_rausch  = np.clip(X_test  + RAUSCH_FAKTOR * np.random.randn(*X_test.shape),  0, 1).astype('float32')

# Convolutional Denoising Autoencoder
eingabe = keras.Input(shape=(28, 28, 1))
# Encoder
x = keras.layers.Conv2D(32, (3,3), activation='relu', padding='same')(eingabe)
x = keras.layers.MaxPooling2D((2,2), padding='same')(x)
x = keras.layers.Conv2D(16, (3,3), activation='relu', padding='same')(x)
x = keras.layers.MaxPooling2D((2,2), padding='same')(x)
# Decoder
x = keras.layers.Conv2D(16, (3,3), activation='relu', padding='same')(x)
x = keras.layers.UpSampling2D((2,2))(x)
x = keras.layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
x = keras.layers.UpSampling2D((2,2))(x)
ausgabe = keras.layers.Conv2D(1, (3,3), activation='sigmoid', padding='same')(x)

denoiser = keras.Model(eingabe, ausgabe, name='DenoisingAutoencoder')
denoiser.compile(optimizer='adam', loss='binary_crossentropy')
denoiser.summary()

# Training: verrauschte Bilder als Input, saubere als Ziel!
history = denoiser.fit(X_train_rausch, X_train,
                        epochs=10, batch_size=256,
                        validation_split=0.1, verbose=1)

# Denoising Ergebnisse
X_denoised = denoiser.predict(X_test_rausch[:8], verbose=0)

fig, axes = plt.subplots(3, 8, figsize=(16, 6))
for i in range(8):
    axes[0, i].imshow(X_test[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', rotation=0, labelpad=40, fontsize=9)

    axes[1, i].imshow(X_test_rausch[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Verrauscht', rotation=0, labelpad=40, fontsize=9)

    axes[2, i].imshow(X_denoised[i].squeeze(), cmap='gray')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('Entrauscht', rotation=0, labelpad=40, fontsize=9)

plt.suptitle(f'Denoising Autoencoder (Rausch-Faktor={RAUSCH_FAKTOR})')
plt.tight_layout()
plt.savefig('denoising_autoencoder.png', dpi=150)
plt.show()
